# CogniVoice Component D - Colab training

Runs feature extraction (slow, GPU) + fusion model training (fast).

**Before running:**
1. Runtime -> Change runtime type -> **T4 GPU**
2. Your Google Drive must contain:
   - `MyDrive/cognivoice/meld_wav/train|val|test/*.wav`  (the MELD wavs)
   - `MyDrive/cognivoice/metadata_meld.csv`

Run cells top to bottom. Outputs (model + features) are saved back to
Drive so nothing is lost when the session ends.

In [ ]:
# 1. Mount Google Drive (authorise when the popup asks)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/cognivoice'

# Locate the wav folders - handles both meld_wav/train and meld_wav/meld/train
if os.path.isdir(f'{DRIVE}/meld_wav/train'):
    WAV_ROOT = f'{DRIVE}/meld_wav'
elif os.path.isdir(f'{DRIVE}/meld_wav/meld/train'):
    WAV_ROOT = f'{DRIVE}/meld_wav/meld'
else:
    raise AssertionError(
        'meld wav folders not found - expected train/val/test inside '
        'MyDrive/cognivoice/meld_wav (or meld_wav/meld). '
        'Check the folder is named cognivoice and sits directly in My Drive.')

assert os.path.exists(f'{DRIVE}/metadata_meld.csv'), \
    'metadata_meld.csv must be directly inside MyDrive/cognivoice'
print('Drive OK - wavs at', WAV_ROOT)

In [ ]:
# 2. Get the project code and install dependencies (~3 min)
%cd /content
!rm -rf cognivoice-component-d
!git clone -q https://github.com/Prathikesh/cognivoice-component-d.git
%cd cognivoice-component-d
!pip install -q funasr modelscope librosa soundfile praat-parselmouth tqdm
print('setup done')

In [ ]:
# 3. Point the metadata at the Drive wav files
# (the csv was built on the Mac, so its paths need rewriting for Colab)
import pandas as pd
os.makedirs('data', exist_ok=True)   # data/ is gitignored, so create it
os.makedirs('models', exist_ok=True)
meta = pd.read_csv(f'{DRIVE}/metadata_meld.csv')
meta['path'] = meta['path'].str.replace(r'.*/meld/', f'{WAV_ROOT}/', regex=True)
meta.to_csv('data/metadata_meld.csv', index=False)
# sanity: the first file must actually exist in Drive
assert os.path.exists(meta['path'].iloc[0]), meta['path'].iloc[0]
print(len(meta), 'clips, paths OK')

In [ ]:
# 4. FEATURE EXTRACTION - the slow step (~20-40 min on T4)
# Runs every clip through the frozen encoder + prosody extractor once,
# caches everything to one npz. Skipped automatically if already done.
if os.path.exists(f'{DRIVE}/features_meld.npz'):
    print('features already cached in Drive - skipping extraction')
    !cp {DRIVE}/features_meld.npz data/features_meld.npz
else:
    !python scripts/extract_features.py --metadata data/metadata_meld.csv --out data/features_meld.npz --encoder plus_large
    !cp data/features_meld.npz {DRIVE}/features_meld.npz
    print('features cached to Drive')

In [ ]:
# 5. TRAIN the fusion model (minutes)
!python scripts/train_fusion.py --features data/features_meld.npz --out models/fusion_v2.pt

In [ ]:
# 6. Save the trained model + history back to Drive
!cp models/fusion_v2.pt {DRIVE}/fusion_v2.pt
!cp models/fusion_v2.history.json {DRIVE}/fusion_v2.history.json
print('saved to Drive: fusion_v2.pt + history')
print('Download fusion_v2.pt from Drive and place it in models/ on your Mac.')